In [ ]:
import qsharp
import qsharp_widgets
from qsharp_widgets import Histogram
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
print(f"qsharp {__import__("importlib.metadata", fromlist=["version"]).version("qsharp")}")

<h3>GHZ state with full state vector simulator</h3>

In [ ]:
%%qsharp

import Std.Diagnostics.DumpMachine;

operation GHZ(n : Int) : Result[] {
    use qs = Qubit[n];
    ???
    for i in 1..n-1 {
        CNOT(qs[0], qs[i]);
    }
    Message($"{n}-qubit GHZ state:");
    DumpMachine();
    MResetEachZ(qs)
}

GHZ(4)

<h3>Memory scaling demo: 2^n amplitudes, each 16 bytes (complex128)</h3>

In [ ]:
# Memory scaling demo: 2^n amplitudes, each 16 bytes (complex128)
qubit_counts = list(range(5, 31, 5))
memory_gb = ???

plt.figure(figsize=(8, 4))
plt.semilogy(qubit_counts, memory_gb, 'o-')
plt.xlabel("Number of qubits")
plt.ylabel("Memory (GB)")
plt.title("Full state vector: memory vs qubit count")
plt.grid(True, which='both', alpha=0.4)
plt.axhline(8, color='red', linestyle='--', label='8 GB RAM')
plt.axhline(64, color='orange', linestyle='--', label='64 GB RAM')
plt.legend()
plt.tight_layout()
plt.show()
print("State vector becomes impractical around 30 qubits on commodity hardware.")

<h3>DumpMachine and DumpRegister demo</h3>

In [ ]:
%%qsharp

import Std.Diagnostics.*;

operation DumpDemo() : Unit {
    use q = Qubit[2];
    H(q[0]);
    // q[0] is in |+⟩, q[1] is still |0⟩ — separable
    ???
    DumpMachine();

    Message("Sub-state of q[0] only:");
    DumpRegister(q[0..0]);   // DumpRegister takes a Qubit[]

    CNOT(q[0], q[1]);
    Message("After CNOT — entangled Bell state:");
    DumpMachine();
    ResetAll(q);
}

DumpDemo()

<h3>This is a purely Clifford circuit — H, CNOT, S, X, Z, measurements</h3>

In [ ]:
%%qsharp

// This is a purely Clifford circuit — H, CNOT, S, X, Z, measurements
operation CliffordCircuit(n : Int) : Result[] {
    use qs = Qubit[n];
    // Create a large GHZ state with only Clifford gates
    ???
    for i in 1..n-1 {
        CNOT(qs[0], qs[i]);
    }
    // Apply S gates (Clifford)
    for q in qs { S(q); }
    // Undo the S gates
    for q in qs { Adjoint S(q); }
    MResetEachZ(qs)
}

<h3>Run a large Clifford circuit — simulatable even at 100+ qubits</h3>

In [ ]:
# Run a large Clifford circuit — simulatable even at 100+ qubits
import time

for n in [10, 30, 60]:
    start = time.time()
    results = qsharp.run(f"CliffordCircuit({n})", 1)
    elapsed = ???
    print(f"n={n:3d} qubits: {elapsed*1000:.1f} ms")

<h3>BellPair operation for noise experiments</h3>

In [ ]:
%%qsharp

operation BellPair() : Result[] {
    use q = Qubit[2];
    ???
    CNOT(q[0], q[1]);
    MResetEachZ(q)
}

<h3>Ideal vs depolarizing noise comparison</h3>

In [ ]:
N_SHOTS = 500

# Ideal
ideal = qsharp.run("BellPair()", N_SHOTS)
# Depolarizing at 5%
noisy = qsharp.run("BellPair()", N_SHOTS, noise=qsharp.DepolarizingNoise(0.05))

def count_errors(results):
    """Count shots where qubits disagree (error indicator for Bell state)"""
    errors = ???
    return errors / len(results)

print(f"Ideal error rate:    {count_errors(ideal):.3f}")
print(f"5% depol error rate: {count_errors(noisy):.3f}")

<h3>Cat5 operation for noise sweep</h3>

In [ ]:
%%qsharp

operation Cat5() : Result[] {
    use q = Qubit[5];
    H(q[0]);
    ???
    MResetEachZ(q)
}

<h3>Sweep bit-flip noise probabilities and show histograms</h3>

In [ ]:
# Sweep bit-flip noise probabilities and show histograms
noise_probs = [0.0, 0.01, 0.05, 0.15]
for p in noise_probs:
    if p == 0.0:
        result = qsharp.run("Cat5()", 500)
    else:
        result = ???
    counts = Counter(str(r) for r in result)
    display(f"Bit-flip noise p={p}: top outcomes = {counts.most_common(5)}")

<h3>Qubit loss simulation with Histogram</h3>

In [ ]:
result = qsharp.run("BellPair()", 200, qubit_loss=0.3)
???
print("Note 'Loss' entries in the histogram — qubits lost during simulation.")

<h3>Safe handling of qubit loss at runtime</h3>

In [ ]:
%%qsharp

// Safe handling of qubit loss at runtime
operation SafeMeasure() : (Bool, Bool) {
    use q = Qubit();
    H(q);
    let res = MResetZ(q);
    if ??? {
        return (true, false);   // (was_lost, was_one)
    } elif res == One {
        return (false, true);
    } else {
        return (false, false);
    }
}

SafeMeasure()

<h3>Count lost qubits across multiple shots</h3>

In [ ]:
loss_results = qsharp.run("SafeMeasure()", 100, qubit_loss=0.4)
lost = ???
print(f"Lost {lost}/100 qubits at qubit_loss=0.40")

<h3>Pure Y noise: py=5%, px=pz=0</h3>

In [ ]:
# Pure Y noise: py=5%, px=pz=0
result_y = ???
display(Histogram(result_y))

# Mixed X+Z noise (equivalent to depolarizing approx)
result_xz = qsharp.run("Cat5()", 1000, noise=(0.025, 0.0, 0.025))
display(Histogram(result_xz))